# KCA full-scale reproduction (Colab): MSR-Action3D · official width · 2048 points per frame

> **English copy of the executed notebook.** Text, comments and printed messages were translated from Chinese;
> the code is unchanged and the outputs are the ones recorded in the actual run (Oct 21–22, 2026, Colab A100).

This notebook does three things on a Colab GPU:
1. Trains P4Transformer, PSTNet and Mamba4D from scratch with the **official configuration** (official width, 2048 points per frame, 24 frames);
2. Re-runs every real-data experiment of the paper: the P4T main benchmark (100 clips × 15 methods), the cross-backbone comparison (40 clips each on PSTNet / Mamba4D), four crossover experiments and two control variants, and the ablation / sanity check / displace probe;
3. **Post-processing**: regenerates every table, number macro and figure in a copy of the paper, `paper_full/`, compiles the PDF and checks the page count, then writes an item-by-item comparison report of the laptop-scale vs full-scale results (which claims still hold, which sentences need rewriting).

The original `paper/` and the laptop-scale results are not modified.

### Preparation (once)
1. On the Mac, run `bash ~/Documents/4DXAI/colab/make_bundle.sh`. It writes two files to `~/kca_work/colab_upload/`: `kca_bundle.tar.gz` (code + paper sources + the three official repositories, about 4 MB) and `msr_npz.tar.gz` (MSR point-cloud data, about 450 MB).
2. Upload both files to Google Drive at `MyDrive/kca_full/upload/`.
3. Open this notebook in Colab and choose **Runtime → Change runtime type → A100 GPU** (or L4).

### How to run
Run from top to bottom (or "Run all"). Checkpoints, per-clip results and logs are written straight to `MyDrive/kca_full/` on Drive. **After a disconnect, reconnect and run everything again from the top**: finished steps are skipped automatically, training resumes from the latest evaluation (every 5 epochs), and the benchmark resumes from the next clip. Keep the browser tab open while it runs.

### Time (rough estimate, A100)
| Step | Time |
|---|---|
| Environment (mamba_ssm is compiled on the first run, then the cached wheel on Drive is used) | 5–60 minutes |
| Training P4T / PSTNet / Mamba4D | about 1 / 1–2 / 2–4 hours |
| Explanation benchmark + crossover + ablation | about 3–5 hours |
| Post-processing | about 10 minutes |

The speed-test cell in Section 3 gives a real estimate of the training time on the GPU you were assigned. An A100 uses compute units quickly: check your balance before starting, or split the run over several sessions.

## 0. Configuration

In [ ]:
# ===== Configuration: usually the only cell you need to edit =====
import os, sys, glob, json, shutil, subprocess, time

DRIVE_ROOT = "/content/drive/MyDrive/kca_full"   # checkpoints, results and logs are all written here (survive disconnects)
UPLOAD = f"{DRIVE_ROOT}/upload"                  # put the two files produced by make_bundle.sh here
POINTS = 2048                                    # points per frame (the setting of all three official repositories)

# Training recipe (delete the corresponding ckpt directory before switching recipes)
#   "adamw": the recipe the paper currently describes (AdamW 3e-4, weight decay 0.05, cosine after 5 warm-up epochs)
#   "sgd"  : the official recipe of each repository (SGD 0.01, momentum 0.9, weight decay 1e-4, 10 warm-up epochs, decay 0.1 at epochs 20/30)
RECIPE = "adamw"
CPV = 17      # random 24-frame clips drawn per training video per epoch; 17 ≈ the clips the official loader enumerates per epoch (17.4 on average)
TRAIN = {     # ckpt directory name, batch size (official value), epochs for each recipe (the sgd column is the official value)
    "p4transformer": dict(dir="p4t",     bs=14, adamw=50, sgd=50),
    "pstnet":        dict(dir="pstnet",  bs=16, adamw=35, sgd=35),
    "mamba4d":       dict(dir="mamba4d", bs=16, adamw=50, sgd=100),
}
PUBLISHED = {"p4transformer": 90.9, "pstnet": 91.2, "mamba4d": 92.7}   # published accuracy (MSR, 24 frames, 2048 points)
OTHER_METHODS = "random,integrated_gradients,point_drop,occam,extremal,kca,kca_ctx"   # methods run on PSTNet / Mamba4D (as in the paper)
RUN_TESTS = True   # run the 7 CPU unit tests once after unpacking (about 5 minutes)

## 1. Mount Drive, unpack code and data

In [ ]:
# [colab-only]
from google.colab import drive
drive.mount("/content/drive")
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → choose A100 or L4"
p = torch.cuda.get_device_properties(0)
print(f"GPU {p.name} {p.total_memory / 1e9:.0f} GB | torch {torch.__version__} | CUDA {torch.version.cuda} | "
      f"Python {sys.version.split()[0]} | {os.cpu_count()} CPUs")
for f in ("kca_bundle.tar.gz", "msr_npz.tar.gz"):
    print(f"{UPLOAD}/{f}:", "✓" if os.path.exists(f"{UPLOAD}/{f}") else "✗ not found")
assert os.path.exists(f"{UPLOAD}/kca_bundle.tar.gz"), "Run colab/make_bundle.sh on the Mac first, then upload kca_bundle.tar.gz to the folder above"

Mounted at /content/drive
GPU NVIDIA A100-SXM4-40GB 42 GB | torch 2.11.0+cu130 | CUDA 13.0 | Python 3.13.15 | 12 CPUs
/content/drive/MyDrive/kca_full/upload/kca_bundle.tar.gz: ✓
/content/drive/MyDrive/kca_full/upload/msr_npz.tar.gz: ✓


In [ ]:
import torch
W = "/content/kca"                       # local working directory (unpacked again in every session; local disk is faster than Drive)
CODE, PAPER, EXT, DATA = f"{W}/code", f"{W}/paper", f"{W}/ext", f"{W}/data/msr"
CK, RES, LOGS, REP = (f"{DRIVE_ROOT}/{d}" for d in ("ckpt", "results/msr", "logs", "report"))
PF = f"{DRIVE_ROOT}/paper_full"          # copy of the paper whose tables / number macros / figures use the full-scale results
for d in (CK, RES, LOGS, REP):
    os.makedirs(d, exist_ok=True)

PY = sys.executable
ENV = dict(os.environ, PYTHONPATH=f"{CODE}/src", PYTHONDONTWRITEBYTECODE="1", PYTHONUNBUFFERED="1",
           MPLBACKEND="Agg", KCA_EXT=EXT,
           KCA_PURE_OPS="1",           # official model code + the pure-PyTorch operators of cpu_ops (same semantics as the laptop version),
                                       # plus two verified CUDA fast paths enabled automatically (FPS, selective scan)
           KCA_MAMBA_CHECKPOINT="1")   # recompute Mamba layers in the backward pass (bit-identical values, less GPU memory)
if os.path.isdir("/usr/local/cuda"):
    ENV.setdefault("CUDA_HOME", "/usr/local/cuda")


def run(cmd, log=None, **env):
    """Run a command in code/: stream its output live and append it to logs/<log> on Drive; raise on failure."""
    print("$", cmd, flush=True)
    t0, fh = time.time(), (open(f"{LOGS}/{log}", "a") if log else None)
    p = subprocess.Popen(cmd, shell=True, cwd=CODE, env={**ENV, **env}, text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    try:
        for line in p.stdout:
            print(line, end="", flush=True)
            if fh:
                fh.write(line)
                fh.flush()
        p.wait()
    except KeyboardInterrupt:            # when the cell is interrupted by hand, stop the child process too
        p.kill()
        raise
    finally:
        if fh:
            fh.close()
    print(f"[exit {p.returncode} · {(time.time() - t0) / 60:.1f} min]", flush=True)
    if p.returncode:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd}")


def n_lines(path):
    return sum(1 for line in open(path) if line.strip()) if os.path.exists(path) else 0


if not os.path.exists(f"{CODE}/src/kca"):
    os.makedirs(W, exist_ok=True)
    subprocess.run(["tar", "-xzf", f"{UPLOAD}/kca_bundle.tar.gz", "-C", W], check=True)
if len(glob.glob(f"{DATA}/*.npz")) < 567:
    os.makedirs(f"{W}/data", exist_ok=True)
    if os.path.exists(f"{UPLOAD}/msr_npz.tar.gz"):
        subprocess.run(["tar", "-xzf", f"{UPLOAD}/msr_npz.tar.gz", "-C", f"{W}/data"], check=True)
    else:   # fallback: download raw MSR-Action3D (the link provided by MeteorNet, 64.6 MB) and convert it; the conversion is deterministic
        raw = f"{W}/data/msr_raw"
        os.makedirs(raw, exist_ok=True)
        subprocess.run("apt-get -qq install -y libarchive-tools > /dev/null", shell=True, check=True)
        subprocess.run([PY, "-m", "gdown", "1djwAK3oZTAIFbCz531eClxINmsZgGO_H", "-O", f"{raw}/msr.zip"], check=True)
        subprocess.run('unzip -o -q msr.zip && mkdir -p depth skel2d && bsdtar -xf Depth.rar -C depth '
                       '&& bsdtar -xf "MSRAction3DSkeleton(20joints).rar" -C skel2d', shell=True, cwd=raw, check=True)
        run(f'{PY} -m kca.data.prepare_msr --depth {raw}/depth/Depth '
            f'--skeleton "{raw}/skel2d/MSRAction3DSkeleton(20joints)" --out {DATA}')
n = len(glob.glob(f"{DATA}/*.npz"))
assert n >= 567, f"Only {n} npz files (expected 567)"
print(f"Code: {CODE}\nData: {n} npz files\nOfficial repositories (commit):\n" + open(f"{EXT}/COMMITS.txt").read())

Code: /content/kca/code
Data: 567 npz files
Official repositories (commit):
P4Transformer 461b83e9ac8ffd5653e09db03e23c56fc8bd5ab2
Point-Spatio-Temporal-Convolution 04f9b47bce1907e28f3f3862dda047178c5e81dd
Mamba4D 8d5bbe20f6ecf8124f56a4e55c8a1aa9303d6fe4



In [ ]:
# [colab-only]
subprocess.run([PY, "-m", "pip", "install", "-q", "einops", "timm", "termcolor", "ninja", "pypdf"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'einops', 'timm', 'termcolor', 'ninja', 'pypdf'], returncode=0)

In [ ]:
if RUN_TESTS:   # the same 7 tests as step 2 of reproduce.sh on the Mac, forced onto the CPU
    for t in ("test_decomposition", "test_symmetrization", "test_correspondence_free", "test_metrics_baselines",
              "test_pipeline_e2e", "test_real_backbone", "test_displace_probe"):
        run(f"{PY} tests/{t}.py", log="tests.log", CUDA_VISIBLE_DEVICES="", KCA_PURE_OPS="0")

$ /usr/bin/python3 tests/test_decomposition.py
OK  Smot(A)=4.800 Sgeo(A)=16.000 | Smot(B)=0.000 Sgeo(B)=16.000 | eps=0.00e+00
OK  explain(): 4 regions scored, 13 forward passes, fields (1, 4, 8)

ALL CORE TESTS PASSED
[exit 0 · 0.1 min]
$ /usr/bin/python3 tests/test_symmetrization.py
OK  vertices | phi=13.702 = frz(shape)=0.902 + geo(motion)=12.800 - abl=0.000
OK  symmetrized completeness holds; additive Delta_int=1.91e-06 (~0)
OK  interacting oracle | completeness exact, Delta_int=11.549 (!=0, detected)

ALL SYMMETRIZATION TESTS PASSED
[exit 0 · 0.0 min]
$ /usr/bin/python3 tests/test_correspondence_free.py
OK  per-frame point shuffling leaves KCA scores unchanged (max diff 3.0e-07)
OK  freeze/displace/abl-geo act on the tracked region without point identity; index-aligned freeze fails the same check (drift 0.43)

ALL CORRESPONDENCE-FREE TESTS PASSED
[exit 0 · 0.0 min]
$ /usr/bin/python3 tests/test_metrics_baselines.py
OK  all 17 methods run; attention methods correctly n/a on SSM orac

## 2. CUDA fast paths

The official model code is not modified; its CUDA-only operators are replaced by the pure-PyTorch versions in `kca/backbones/cpu_ops.py`, with the same semantics as the laptop-scale version. Two of them are step-by-step loops that are too slow at official scale:
- **FPS**: PSTNet needs about 33,000 steps per forward pass. Here a single-kernel version is compiled that picks points by the same rule as the loop (start from point 0, break ties by the smallest index; unlike the official kernel, it does not skip points near the origin).
- **Mamba's selective scan**: the official-width Mamba4D scans tens of thousands of steps per forward pass, and the loop stores the state of every step, which does not fit in GPU memory. Here the compiled `selective_scan_cuda` kernel of `mamba_ssm` is called.

Before either is enabled, it is compared with the loop on random inputs; if the check fails, the loop is used automatically (same results, only slower). The `mamba_ssm` wheel is cached on Drive under `wheels/` after the first build, and later sessions install it directly.

In [ ]:
# [colab-only]
def ssc_ok():
    return subprocess.run([PY, "-c", "import torch, selective_scan_cuda"], capture_output=True).returncode == 0

tag = f"torch{torch.__version__.split('+')[0]}_cu{torch.version.cuda}_py{sys.version_info.major}{sys.version_info.minor}"
WHEELS = f"{DRIVE_ROOT}/wheels/{tag}"
if not ssc_ok():
    os.makedirs(WHEELS, exist_ok=True)
    whl = glob.glob(f"{WHEELS}/mamba_ssm-*.whl")
    if not whl:
        print("First run: building the mamba_ssm wheel and caching it on Drive. A few minutes if a matching prebuilt wheel exists, otherwise a source build of about 20–60 minutes.")
        run(f'MAX_JOBS={os.cpu_count()} {PY} -m pip wheel --no-build-isolation --no-deps mamba-ssm -w "{WHEELS}"',
            log="mamba_ssm_build.log")
        whl = glob.glob(f"{WHEELS}/mamba_ssm-*.whl")
    run(f'{PY} -m pip install -q --no-deps "{whl[0]}"')
print("selective_scan_cuda:", "available ✓" if ssc_ok() else "not available ✗ (see the next cell)")

First run: building the mamba_ssm wheel and caching it on Drive. A few minutes if a matching prebuilt wheel exists, otherwise a source build of about 20–60 minutes.
$ MAX_JOBS=12 /usr/bin/python3 -m pip wheel --no-build-isolation --no-deps mamba-ssm -w "/content/drive/MyDrive/kca_full/wheels/torch2.11.0_cu13.0_py313"
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 8.1 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.2.post1-cp313-cp313-linux_x86_64.whl size=349376262 sha256=7fbb797c09736a6d7709e48e40f46206099101b14a976ed18a1943e68a175a3f
  Stored in directory: /root/.cache/pip/wheels/7f/68/c4/588f5c6c8950032840738453efc028cde9be35603c72d662ca
Successfully built mamba-ssm
[exit 0 · 17.6 min]
$ /usr/bin/python3 -m pip install -q --no-deps "/content/drive/MyDrive/kca_full/wheels/torch2.11.0_cu13.0_py313/mamba_ssm-2.3.2.post1-cp313-cp313-linux_

In [ ]:
# [colab-only]
r = subprocess.run([PY, "-c", "from kca.backbones import cpu_ops; cpu_ops.install(); "
                    "from kca.backbones import cuda_fast; import json; print(json.dumps(cuda_fast.STATUS))"],
                   cwd=CODE, env=ENV, capture_output=True, text=True)
print(r.stderr[-3000:])
if r.returncode:
    raise RuntimeError("The fast-path check itself failed; see the output above")
STATUS = json.loads(r.stdout.strip().splitlines()[-1])
FAST_FPS, FAST_SCAN = STATUS["fps"] == "on", STATUS["scan"] == "on"
print("FPS kernel:", STATUS["fps"], "| selective scan kernel:", STATUS["scan"])
if not FAST_FPS:
    print("⚠️ FPS falls back to the step-by-step loop: same results, but PSTNet (1024 anchors per frame) becomes much slower.")
if not FAST_SCAN:
    print("⚠️ No selective scan kernel: with the step-by-step loop the official-width Mamba4D is too slow and does not fit in GPU memory, so every Mamba4D step below is skipped.")

[kca.cuda_fast] fps: on (picks equal to the loop's: 100.00%) | scan: on

FPS kernel: on | selective scan kernel: on


## 3. Speed test (optional)

For each backbone, runs a few training steps on random input, estimates the training time for the Section 0 configuration (excluding the test-set evaluation every 5 epochs), and times one explanation forward pass of 32 samples.

In [ ]:
# [colab-only]
import numpy as np
SPEED = r"""
import json, sys, time, torch, torch.nn.functional as F
from kca.backbones import cpu_ops
from kca.backbones.official import build_net, preprocess
name, bs = sys.argv[1], int(sys.argv[2])
torch.backends.cuda.matmul.allow_tf32 = True           # same as the training command
net = build_net(name, 20).cuda().train()
opt = torch.optim.AdamW(net.parameters(), lr=1e-5)
x = (torch.rand(bs, 24, 2048, 3, device="cuda") - 0.5) * 4
y = torch.randint(0, 20, (bs,), device="cuda")
def step():
    with cpu_ops.device_compat():
        loss = F.cross_entropy(net(preprocess(x, {"center": True})), y)
    opt.zero_grad(); loss.backward(); opt.step()
step(); torch.cuda.synchronize(); t = time.time()
for _ in range(3):
    step()
torch.cuda.synchronize(); tr = (time.time() - t) / 3
peak = torch.cuda.max_memory_allocated() / 1e9
torch.backends.cuda.matmul.allow_tf32 = False          # explanations run in FP32
net.eval(); xe = x[:1].expand(32, -1, -1, -1).contiguous()
with torch.no_grad(), cpu_ops.device_compat():
    net(preprocess(xe, {"center": True})); torch.cuda.synchronize(); t = time.time()
    net(preprocess(xe, {"center": True})); torch.cuda.synchronize()
print(json.dumps({"train_s_it": tr, "fwd32_s": time.time() - t, "peak_GB": peak}))
"""
open(f"{W}/speed.py", "w").write(SPEED)
n_vid = sum(1 for f in glob.glob(f"{DATA}/*.npz") if int(os.path.basename(f).split("_")[1][1:]) <= 5
            and len(np.load(f, allow_pickle=True)["point_clouds"]) >= 24)
for bb, c in TRAIN.items():
    if bb == "mamba4d" and not FAST_SCAN:
        continue
    r = subprocess.run([PY, f"{W}/speed.py", bb, str(c["bs"])], cwd=CODE, env=ENV, capture_output=True, text=True)
    if r.returncode:
        print(bb, "speed test failed:\n", r.stderr[-2000:])
        continue
    s = json.loads(r.stdout.strip().splitlines()[-1])
    it = c[RECIPE] * -(-n_vid * CPV // c["bs"])
    print(f"{bb:14s} {s['train_s_it']:.2f} s/iter, peak GPU memory {s['peak_GB']:.1f} GB → {RECIPE} {c[RECIPE]} epochs = "
          f"{it} iter ≈ {it * s['train_s_it'] / 3600:.1f} h | explanation: one forward pass of 32 samples {s['fwd32_s']:.2f} s")

p4transformer  0.12 s/iter, peak GPU memory 5.6 GB → adamw 50 epochs = 15650 iter ≈ 0.5 h | explanation: one forward pass of 32 samples 0.24 s
pstnet         0.45 s/iter, peak GPU memory 20.7 GB → adamw 35 epochs = 9590 iter ≈ 1.2 h | explanation: one forward pass of 32 samples 0.57 s
mamba4d        2.43 s/iter, peak GPU memory 20.1 GB → adamw 50 epochs = 13700 iter ≈ 9.2 h | explanation: one forward pass of 32 samples 1.67 s


## 4. Train the three backbones (official width, 2048 points per frame)

One cell per backbone; they can be run separately. Every 5 epochs the model is evaluated on the test set and `last.pth` is saved; the paper uses the last epoch and does not select models on the test set. After an interruption, re-run the same cell and it continues from the latest save (`--resume`). Training uses TF32 matrix multiplication; explanation and evaluation still use FP32.

In [ ]:
def train(bb):
    c = TRAIN[bb]
    out, ep = f"{CK}/{c['dir']}", c[RECIPE]
    if os.path.exists(f"{out}/summary.json"):
        print(f"{bb}: already trained, skipping ({out}/summary.json)")
        return
    if RECIPE == "sgd":   # official decay at epochs 20 and 30; train_official takes them as fractions of all epochs (+0.5 avoids rounding errors)
        opt = f"--opt sgd --warmup 10 --milestones {20.5 / ep:.6f} {30.5 / ep:.6f}"
    else:
        opt = "--opt adamw"
    run(f"{PY} -m kca.run.train_official --backbone {bb} --scale official --points {POINTS} --data {DATA} "
        f"--out {out} --epochs {ep} --clips-per-video {CPV} --bs {c['bs']} {opt} --device cuda --resume",
        log=f"train_{bb}.log", TORCH_ALLOW_TF32_CUBLAS_OVERRIDE="1")


train("p4transformer")

$ /usr/bin/python3 -m kca.run.train_official --backbone p4transformer --scale official --points 2048 --data /content/kca/data/msr --out /content/drive/MyDrive/kca_full/ckpt/p4t --epochs 50 --clips-per-video 17 --bs 14 --opt adamw --device cuda --resume
[kca.cuda_fast] fps: on (picks equal to the loop's: 100.00%) | scan: on
[p4transformer] device=cuda cfg={} params=44.14M train=270 test=297 T=24 N=2048
ep   1/50 loss 1.896 train-acc 0.441 lr 6.221e-05 (48s)
ep   2/50 loss 0.391 train-acc 0.882 lr 0.0001216 (48s)
ep   3/50 loss 0.386 train-acc 0.875 lr 0.000181 (47s)
ep   4/50 loss 0.252 train-acc 0.921 lr 0.0002404 (48s)
ep   5/50 loss 0.269 train-acc 0.912 lr 0.0002998 (48s)  test-video-acc 0.596
ep   6/50 loss 0.235 train-acc 0.927 lr 0.0002996 (48s)
ep   7/50 loss 0.116 train-acc 0.962 lr 0.0002986 (48s)
ep   8/50 loss 0.092 train-acc 0.972 lr 0.0002968 (48s)
ep   9/50 loss 0.065 train-acc 0.982 lr 0.0002943 (48s)
ep  10/50 loss 0.085 train-acc 0.972 lr 0.0002911 (48s)  test-video-ac

In [ ]:
train("pstnet")

$ /usr/bin/python3 -m kca.run.train_official --backbone pstnet --scale official --points 2048 --data /content/kca/data/msr --out /content/drive/MyDrive/kca_full/ckpt/pstnet --epochs 35 --clips-per-video 17 --bs 16 --opt adamw --device cuda --resume
[kca.cuda_fast] fps: on (picks equal to the loop's: 100.00%) | scan: on
[pstnet] device=cuda cfg={} params=8.26M train=270 test=297 T=24 N=2048
ep   1/35 loss 2.036 train-acc 0.442 lr 6.218e-05 (136s)
ep   2/35 loss 0.476 train-acc 0.870 lr 0.0001216 (135s)
ep   3/35 loss 0.253 train-acc 0.929 lr 0.000181 (135s)
ep   4/35 loss 0.266 train-acc 0.916 lr 0.0002404 (135s)
ep   5/35 loss 0.227 train-acc 0.925 lr 0.0002998 (135s)  test-video-acc 0.718
ep   6/35 loss 0.260 train-acc 0.923 lr 0.0002992 (135s)
ep   7/35 loss 0.183 train-acc 0.945 lr 0.0002968 (135s)
ep   8/35 loss 0.192 train-acc 0.942 lr 0.0002927 (135s)
ep   9/35 loss 0.180 train-acc 0.951 lr 0.0002872 (135s)
ep  10/35 loss 0.132 train-acc 0.962 lr 0.0002801 (135s)  test-video-acc 

In [ ]:
if FAST_SCAN:
    train("mamba4d")
else:
    print("Skipping Mamba4D (no selective scan kernel)")

$ /usr/bin/python3 -m kca.run.train_official --backbone mamba4d --scale official --points 2048 --data /content/kca/data/msr --out /content/drive/MyDrive/kca_full/ckpt/mamba4d --epochs 50 --clips-per-video 17 --bs 16 --opt adamw --device cuda --resume
[kca.cuda_fast] fps: on (picks equal to the loop's: 100.00%) | scan: on
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
[mamba4d] device=cuda cfg={} params=109.81M train=270 test=297 T=24 N=2048
[resume] from epoch 10
ep  11/50 loss 0.096 train-acc 0.973 lr 0.0002872 (665s)
ep  12/50 loss 0.079 train-acc 0.974 lr 0.0002826 (668s)
ep  13/50 loss 0.104 train-acc 0.968 lr 0.0002775 (673s)
ep  14/50 loss 0.050 train-acc 0.984 lr 0.0002717 (670s)
ep  15/50 loss 0.204 train-acc 0.938 lr 0.0002653 (669s)  test-video-acc 0.8

In [ ]:
for bb, c in TRAIN.items():
    f = f"{CK}/{c['dir']}/summary.json"
    if not os.path.exists(f):
        print(f"{bb:14s} not finished")
        continue
    s = json.load(open(f))
    last, best = 100 * s["last_acc"], 100 * s["best_acc"]
    warn = "" if last >= PUBLISHED[bb] - 2 else "  ⚠️ more than 2 points below the published value"
    print(f"{bb:14s} last epoch {last:.1f}% (best {best:.1f}%)  published {PUBLISHED[bb]}%  params {s['params_M']:.1f}M{warn}")
print("\nIf the gap is too large: change RECIPE or add epochs, delete the corresponding ckpt directory and re-run that backbone's training cell.")

p4transformer  last epoch 89.2% (best 89.2%)  published 90.9%  params 44.1M
pstnet         last epoch 88.9% (best 88.9%)  published 91.2%  params 8.3M  ⚠️ more than 2 points below the published value
mamba4d        last epoch 88.2% (best 89.2%)  published 92.7%  params 109.8M  ⚠️ more than 2 points below the published value

If the gap is too large: change RECIPE or add epochs, delete the corresponding ckpt directory and re-run that backbone's training cell.


## 5. Explanation benchmark

As in the paper: on P4T, 5 test clips per class (100 in total), 15 methods, all metrics; on PSTNet / Mamba4D, 2 per class (40 in total, the first 40 of the P4T clips), 7 methods, deletion / insertion metrics. Each clip is written to `raw.jsonl` as soon as it finishes; after an interruption the run continues from the next clip.

In [ ]:
def bench(bb, per_class, extra=""):
    out, n = f"{RES}/{bb}", 20 * per_class
    if n_lines(f"{out}/raw.jsonl") >= n:
        print(f"{bb}: all {n} clips done")
        return
    run(f"{PY} -m kca.run.msr_benchmark --backbone {bb} --ckpt {CK}/{TRAIN[bb]['dir']}/last.pth --data {DATA} "
        f"--out {out} --per-class {per_class} --points {POINTS} --device cuda --threads 4 {extra}",
        log=f"bench_{bb}.log")


bench("p4transformer", 5)

$ /usr/bin/python3 -m kca.run.msr_benchmark --backbone p4transformer --ckpt /content/drive/MyDrive/kca_full/ckpt/p4t/last.pth --data /content/kca/data/msr --out /content/drive/MyDrive/kca_full/results/msr/p4transformer --per-class 5 --points 2048 --device cuda --threads 4 
[kca.cuda_fast] fps: on (picks equal to the loop's: 100.00%) | scan: on
[p4transformer] 100 clips (0 done), methods=['random', 'geometry_edge', 'motion_prior', 'grad_x_input', 'smoothgrad', 'integrated_gradients', 'grad_cam_3d', 'point_drop', 'occam', 'extremal', 'attn_rollout', 'transformer_attr', 'staa4d', 'kca', 'kca_ctx']
  [1/100] a01_s06_e01_sdepth.npz label 0 pred 0 (explain 14.9s)
  [2/100] a01_s06_e02_sdepth.npz label 0 pred 0 (explain 11.0s)
  [3/100] a01_s06_e03_sdepth.npz label 0 pred 0 (explain 10.9s)
  [4/100] a01_s07_e01_sdepth.npz label 0 pred 0 (explain 10.9s)
  [5/100] a01_s07_e02_sdepth.npz label 0 pred 0 (explain 10.8s)
  [6/100] a02_s06_e01_sdepth.npz label 1 pred 1 (explain 10.9s)
  [7/100] a02_

In [ ]:
bench("pstnet", 2, f"--metrics delins --methods {OTHER_METHODS}")

$ /usr/bin/python3 -m kca.run.msr_benchmark --backbone pstnet --ckpt /content/drive/MyDrive/kca_full/ckpt/pstnet/last.pth --data /content/kca/data/msr --out /content/drive/MyDrive/kca_full/results/msr/pstnet --per-class 2 --points 2048 --device cuda --threads 4 --metrics delins --methods random,integrated_gradients,point_drop,occam,extremal,kca,kca_ctx
[kca.cuda_fast] fps: on (picks equal to the loop's: 100.00%) | scan: on
[pstnet] 40 clips (0 done), methods=['random', 'integrated_gradients', 'point_drop', 'occam', 'extremal', 'kca', 'kca_ctx']
  [1/40] a01_s06_e01_sdepth.npz label 0 pred 0 (explain 30.0s)
  [2/40] a01_s06_e02_sdepth.npz label 0 pred 0 (explain 25.9s)
  [3/40] a02_s06_e01_sdepth.npz label 1 pred 1 (explain 25.9s)
  [4/40] a02_s06_e02_sdepth.npz label 1 pred 1 (explain 25.8s)
  [5/40] a03_s06_e01_sdepth.npz label 2 pred 7 (explain 25.9s)
  [6/40] a03_s06_e02_sdepth.npz label 2 pred 6 (explain 26.0s)
  [7/40] a04_s06_e01_sdepth.npz label 3 pred 4 (explain 26.0s)
  [8/40]

In [ ]:
if FAST_SCAN:
    bench("mamba4d", 2, f"--metrics delins --methods {OTHER_METHODS}")

$ /usr/bin/python3 -m kca.run.msr_benchmark --backbone mamba4d --ckpt /content/drive/MyDrive/kca_full/ckpt/mamba4d/last.pth --data /content/kca/data/msr --out /content/drive/MyDrive/kca_full/results/msr/mamba4d --per-class 2 --points 2048 --device cuda --threads 4 --metrics delins --methods random,integrated_gradients,point_drop,occam,extremal,kca,kca_ctx
[kca.cuda_fast] fps: on (picks equal to the loop's: 100.00%) | scan: on
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
[mamba4d] 40 clips (0 done), methods=['random', 'integrated_gradients', 'point_drop', 'occam', 'extremal', 'kca', 'kca_ctx']
  [1/40] a01_s06_e01_sdepth.npz label 0 pred 0 (explain 153.1s)
  [2/40] a01_s06_e02_sdepth.npz label 0 pred 0 (explain 153.8s)
  [3/40] a02_s06_e01_sdepth.npz label 1 pr

## 6. Crossover experiment (the same arm: motion vs shape)

For each backbone, train two binary task models (official width, 2048 points per frame), then explain 40 test clips. P4T additionally gets a second training seed and the two control variants from the appendix. Task models are saved right after training and loaded directly on a re-run (`--reuse`).

In [ ]:
def crossover(name, extra):
    out = f"{RES}/{name}"
    j = f"{out}/crossover.json"
    if os.path.exists(j) and all("kca" in json.load(open(j)).get(t, {}) for t in ("motion", "geometry")):
        print(f"{name}: done")
        return
    run(f"{PY} -m kca.run.msr_crossover --data {DATA} --out {out} --scale official --points {POINTS} "
        f"--device cuda --explain-device cuda --reuse {extra}", log=f"{name}.log")


crossover("crossover", "--backbone p4transformer")
crossover("crossover_seed1", "--backbone p4transformer --seed 1")
crossover("crossover_pstnet", "--backbone pstnet")
if FAST_SCAN:
    crossover("crossover_mamba4d", "--backbone mamba4d")

$ /usr/bin/python3 -m kca.run.msr_crossover --data /content/kca/data/msr --out /content/drive/MyDrive/kca_full/results/msr/crossover --scale official --points 2048 --device cuda --explain-device cuda --reuse --backbone p4transformer
arm-moving clips: train 163 test 200
[kca.cuda_fast] fps: on (picks equal to the loop's: 100.00%) | scan: on
[motion] trained: test acc 0.805 (102s)
[motion] acc 0.805 | KCA arm motion share 0.827 (ctx 0.801) | motion-prior EBPG(arm) 0.28
[geometry] trained: test acc 0.835 (98s)
[geometry] acc 0.835 | KCA arm motion share 0.602 (ctx 0.601) | motion-prior EBPG(arm) 0.28
[exit 0 · 8.2 min]
$ /usr/bin/python3 -m kca.run.msr_crossover --data /content/kca/data/msr --out /content/drive/MyDrive/kca_full/results/msr/crossover_seed1 --scale official --points 2048 --device cuda --explain-device cuda --reuse --backbone p4transformer --seed 1
arm-moving clips: train 163 test 200
[kca.cuda_fast] fps: on (picks equal to the loop's: 100.00%) | scan: on
[motion] trained: t

In [ ]:
def variant(tag, extra, target):
    dst = f"{RES}/crossover/{target}"
    if os.path.exists(dst):
        print(f"{target}: done")
        return
    out = f"{RES}/crossover_var_{tag}"
    os.makedirs(out, exist_ok=True)
    run(f"{PY} -m kca.run.msr_crossover --backbone p4transformer --data {DATA} --out {out} --scale official "
        f"--points {POINTS} --device cuda --explain-device cuda --reuse --tasks geometry {extra}",
        log=f"crossover_var_{tag}.log")
    shutil.copy(f"{out}/crossover.json", dst)


# (1) In the shape task, explain the "original class" (the evidence is the absence of the cue; appendix \NumXThickShare).
#     Training is identical to the geometry model of the main crossover (same seed, same beta), so that model is reused.
assert os.path.exists(f"{RES}/crossover/geometry.pth"), "Run the previous cell (main crossover) first"
os.makedirs(f"{RES}/crossover_var_beta3", exist_ok=True)
if not os.path.exists(f"{RES}/crossover_var_beta3/geometry.pth"):
    shutil.copy(f"{RES}/crossover/geometry.pth", f"{RES}/crossover_var_beta3/geometry.pth")
variant("beta3", "--geo-explain original", "crossover_beta3.json")
# (2) Make the arm thinner (beta = 0.3); at laptop scale P4T cannot learn it (\NumXThinAcc)
variant("thin", "--beta 0.3", "crossover_thin.json")

$ /usr/bin/python3 -m kca.run.msr_crossover --backbone p4transformer --data /content/kca/data/msr --out /content/drive/MyDrive/kca_full/results/msr/crossover_var_beta3 --scale official --points 2048 --device cuda --explain-device cuda --reuse --tasks geometry --geo-explain original
arm-moving clips: train 163 test 200
[kca.cuda_fast] fps: on (picks equal to the loop's: 100.00%) | scan: on
[geometry] acc 0.835 | KCA arm motion share 0.875 (ctx 0.736) | motion-prior EBPG(arm) 0.28
[exit 0 · 2.9 min]
$ /usr/bin/python3 -m kca.run.msr_crossover --backbone p4transformer --data /content/kca/data/msr --out /content/drive/MyDrive/kca_full/results/msr/crossover_var_thin --scale official --points 2048 --device cuda --explain-device cuda --reuse --tasks geometry --beta 0.3
arm-moving clips: train 163 test 200
[kca.cuda_fast] fps: on (picks equal to the loop's: 100.00%) | scan: on
[geometry] trained: test acc 0.500 (105s)
[geometry] acc 0.500 | KCA arm motion share 0.563 (ctx 0.559) | motion-prior

## 7. Ablation, sanity check, displace probe (P4T, 40 clips)

Each ablation variant is saved as soon as it finishes; after an interruption, finished variants are skipped.

In [19]:
ab = f"{RES}/ablation/ablation.json"
A = json.load(open(ab)) if os.path.exists(ab) else {}
if len(A.get("ablation", {})) >= 16 and "sanity" in A and "displace" in A:
    print("Ablation already done")
else:
    run(f"{PY} -m kca.run.msr_ablation --ckpt {CK}/p4t/last.pth --data {DATA} --out {RES}/ablation "
        f"--points {POINTS} --device cuda", log="ablation.log")

$ /usr/bin/python3 -m kca.run.msr_ablation --ckpt /content/drive/MyDrive/kca_full/ckpt/p4t/last.pth --data /content/kca/data/msr --out /content/drive/MyDrive/kca_full/results/msr/ablation --points 2048 --device cuda
[kca.cuda_fast] fps: on (picks equal to the loop's: 100.00%) | scan: on
local_m6 {'deletion_auc': 0.235, 'insertion_auc': 0.748, 'smot_ebpg': 0.468, 'smot_pg': 0.425, 'dint_over_nec': 0.456, 'time_s': 0.26, 'n_forward': 19.0, 'motion_share_moving': 0.646}
ctx_m6 {'deletion_auc': 0.192, 'insertion_auc': 0.735, 'smot_ebpg': 0.443, 'smot_pg': 0.425, 'dint_over_nec': 0.478, 'time_s': 2.269, 'n_forward': 257.0, 'motion_share_moving': 0.647}
local_m12 {'deletion_auc': 0.272, 'insertion_auc': 0.753, 'smot_ebpg': 0.542, 'smot_pg': 0.625, 'dint_over_nec': 0.631, 'time_s': 0.473, 'n_forward': 37.0, 'motion_share_moving': 0.728}
ctx_m12 {'deletion_auc': 0.219, 'insertion_auc': 0.747, 'smot_ebpg': 0.543, 'smot_pg': 0.65, 'dint_over_nec': 0.575, 'time_s': 2.322, 'n_forward': 257.0, 'mot

## 8. Post-processing

1. In the paper copy `MyDrive/kca_full/paper_full/`, regenerate every table, the number macros (`tables/numbers.tex`) and the figures. The controlled study uses synthetic networks and does not depend on the backbone scale, so its existing results are reused.
2. Compile the PDF and check that the main text still fits in 8 pages and that there is no unresolved `??`.
3. Write the comparison report `report/REPORT.md`: whether each claim of the paper holds at laptop scale and at full scale, the key numbers side by side, and where the sentences that need rewriting are. `report/numbers_compare.csv` compares every number macro item by item.
4. Pack everything into `kca_full_outputs.zip`, without the `.pth` files; the checkpoints stay on Drive.

This section can also be run before every experiment has finished: missing numbers show as a red ?? in the PDF and as n/a in the report.

> **Note added in the English copy:** at the time of this run the paper was still in its earlier 8-page CVPR format, compiled with tectonic. The paper now uses the AAAI-27 author kit, whose `aaai2027.sty` refuses XeTeX/tectonic and needs pdfLaTeX, and its main-text limit is 7 pages. Update the compile cell and the page check before re-running this section.

In [20]:
CTRL = f"{DRIVE_ROOT}/results/controlled"
if not os.path.exists(CTRL):
    shutil.copytree(f"{CODE}/results/controlled", CTRL)
if os.path.exists(PF):                    # paper_full is re-copied from the original paper and regenerated every time
    shutil.rmtree(PF)
shutil.copytree(PAPER, PF)
run(f"{PY} -m kca.run.paper_tables --results {RES} --paper {PF} --ckpt-root {CK} --data {DATA}", log="postprocess.log")
run(f"{PY} -m kca.run.paper_figures --results {RES} --controlled {CTRL}/crossover.json --data {DATA} --fig {PF}/fig",
    log="postprocess.log")

$ /usr/bin/python3 -m kca.run.paper_tables --results /content/drive/MyDrive/kca_full/results/msr --paper /content/drive/MyDrive/kca_full/paper_full --ckpt-root /content/drive/MyDrive/kca_full/ckpt --data /content/kca/data/msr
{
 "pstnet": {
  "extremal": {
   "deletion_auc": 0.932,
   "insertion_auc": 0.699,
   "ebpg": 0.165
  },
  "integrated_gradients": {
   "deletion_auc": 0.303,
   "insertion_auc": 0.964,
   "ebpg": 0.436
  },
  "kca": {
   "deletion_auc": 0.262,
   "insertion_auc": 0.747,
   "ebpg": 0.382,
   "smot_ebpg": 0.377
  },
  "kca_ctx": {
   "deletion_auc": 0.217,
   "insertion_auc": 0.717,
   "ebpg": 0.398,
   "smot_ebpg": 0.409
  },
  "occam": {
   "deletion_auc": 0.845,
   "insertion_auc": 0.966,
   "ebpg": 0.166
  },
  "point_drop": {
   "deletion_auc": 0.252,
   "insertion_auc": 0.756,
   "ebpg": 0.38
  },
  "random": {
   "deletion_auc": 0.95,
   "insertion_auc": 0.938,
   "ebpg": 0.166
  },
  "_tests": {}
 },
 "p4transformer": {
  "attn_rollout": {
   "deletion_auc

In [ ]:
TEC = shutil.which("tectonic") or "/content/tectonic"
if not os.path.exists(TEC):     # tectonic: a self-contained LaTeX engine that downloads packages on demand (also used on the Mac at the time)
    subprocess.run("curl --proto '=https' --tlsv1.2 -fsSL https://drop-sh.fullyjustified.net | sh",
                   shell=True, cwd="/content", check=True)
B = "/content/paper_build"
shutil.rmtree(B, ignore_errors=True)
shutil.copytree(PF, B)
r = subprocess.run([TEC, "main.tex"], cwd=B, capture_output=True, text=True)
open(f"{LOGS}/tectonic.log", "w").write(r.stdout + r.stderr)
if r.returncode:
    print(r.stderr[-3000:])
    raise RuntimeError("Compilation failed, see logs/tectonic.log")
shutil.copy(f"{B}/main.pdf", f"{PF}/main.pdf")

import re, logging
from pypdf import PdfReader
logging.disable(logging.WARNING)
pages, unresolved = PdfReader(f"{B}/main.pdf").pages, 0
for i, pg in enumerate(pages):            # same checks as ~/kca_work/tools/check_paper.sh
    t = pg.extract_text() or ""
    unresolved += len(re.findall(r"\[\?\]|\?\?", t))
    marks = [m for m in ("1. Introduction", "4. Experiments", "5. Conclusion", "Future work", "References",
                         "Supplementary Material") if m in t]
    if marks and i < 12:
        print(f"  page {i + 1}: {marks}")
print(f"{len(pages)} pages; unresolved ?? / [?]: {unresolved}  →  {PF}/main.pdf")
print("The main text must end on page 8 (References start on page 9). After rewriting the text, confirm with check_paper.sh on the Mac.")

In [22]:
run(f"{PY} -m kca.run.compare_full --old {PAPER}/tables/numbers.tex --new {PF}/tables/numbers.tex "
    f"--paper {PF} --out {REP}", log="postprocess.log")
from IPython.display import Markdown, display
display(Markdown(open(f"{REP}/REPORT.md").read()))

$ /usr/bin/python3 -m kca.run.compare_full --old /content/kca/paper/tables/numbers.tex --new /content/drive/MyDrive/kca_full/paper_full/tables/numbers.tex --paper /content/drive/MyDrive/kca_full/paper_full --out /content/drive/MyDrive/kca_full/report
[report] 4 claim(s) no longer hold -> /content/drive/MyDrive/kca_full/report/REPORT.md
[exit 0 · 0.0 min]


# KCA: laptop scale vs full scale — comparison of results

## 1. Do the paper's claims still hold?

| Claim | Laptop | Full scale | Values (laptop → full scale) |
|---|---|---|---|
| P4Transformer: accuracy within 2 points of the published value | ✓ | ✓ | `BBaccPFT` 90.6 → 89.2 |
| PSTNet: accuracy within 2 points of the published value | ✗ | **✗** | `BBaccPST` 87.5 → 88.9 |
| Mamba4D: accuracy within 2 points of the published value | ✗ | **✗** | `BBaccMBA` 83.6 → 88.2 |
| P4T: KCA-ctx deletion AUC below region occlusion, significant after Bonferroni correction (p < 0.0125) | ✓ | **✗** | `NumPFTKcaCtxDel` 0.178 → 0.207; `NumPFTOcclDel` 0.219 → 0.250; `NumPKcaCtxOcclDel` {=}0.0043 → {=}0.1 |
| P4Transformer: KCA-ctx deletion AUC significantly below IG (p < 0.05) | ✓ | ✓ | `NumPFTKcaCtxDel` 0.178 → 0.207; `NumPFTIGDel` 0.268 → 0.283; `NumPKcaCtxIGDel` {<}10^{-3} → {=}0.0031 |
| P4Transformer: S_mot EBPG on par with or better than IG (not significantly worse) | ✓ | ✓ | `NumPFTKcaCtxSmotEBPG` 0.54 → 0.54; `NumPFTIGEBPG` 0.58 → 0.58; `NumPKcaCtxIGEBPG` {=}0.093 → {=}0.06 |
| PSTNet: KCA-ctx deletion AUC significantly below IG (p < 0.05) | ✓ | ✓ | `NumPSTKcaCtxDel` 0.191 → 0.217; `NumPSTIGDel` 0.264 → 0.303; `NumPPSTKcaCtxIGDel` {=}0.018 → {=}0.007 |
| PSTNet: S_mot EBPG on par with or better than IG (not significantly worse) | ✓ | ✓ | `NumPSTKcaCtxSmotEBPG` 0.45 → 0.41; `NumPSTIGEBPG` 0.46 → 0.44; `NumPPSTKcaCtxIGEBPG` {=}0.72 → {=}0.62 |
| Mamba4D: KCA-ctx deletion AUC significantly below IG (p < 0.05) | ✓ | **✗** | `NumMBAKcaCtxDel` 0.245 → 0.269; `NumMBAIGDel` 0.318 → 0.329; `NumPMBAKcaCtxIGDel` {<}10^{-3} → {=}0.064 |
| Mamba4D: S_mot EBPG on par with or better than IG (not significantly worse) | ✓ | ✓ | `NumMBAKcaCtxSmotEBPG` 0.49 → 0.48; `NumMBAIGEBPG` 0.48 → 0.50; `NumPMBAKcaCtxIGEBPG` {=}0.33 → {=}0.72 |
| PSTNet: KCA-ctx deletion AUC on par with or better than region occlusion (not significantly worse) | ✓ | ✓ | `NumPSTKcaCtxDel` 0.191 → 0.217; `NumPSTOcclDel` 0.222 → 0.252; `NumPPSTKcaCtxOcclDel` {=}0.55 → {=}0.38 |
| Mamba4D: KCA-ctx deletion AUC on par with or better than region occlusion (not significantly worse) | ✓ | ✓ | `NumMBAKcaCtxDel` 0.245 → 0.269; `NumMBAOcclDel` 0.240 → 0.246; `NumPMBAKcaCtxOcclDel` {=}0.46 → {=}0.83 |
| Crossover P4Transformer: motion share of the motion task above that of the shape task, 95% CI of the gap excludes 0 | ✓ | ✓ | `NumXMotShare` 0.81 → 0.83; `NumXGeoShare` 0.39 → 0.60; `NumXGapLo` 0.31 → 0.14; `NumXGapHi` 0.52 → 0.32 |
| Crossover PSTNet: motion share of the motion task above that of the shape task, 95% CI of the gap excludes 0 | ✓ | ✓ | `NumXPstMotShare` 0.85 → 0.86; `NumXPstGeoShare` 0.23 → 0.30; `NumXPstGapLo` 0.55 → 0.48; `NumXPstGapHi` 0.68 → 0.65 |
| Crossover Mamba4D: motion share of the motion task above that of the shape task, 95% CI of the gap excludes 0 | ✓ | ✓ | `NumXMbaMotShare` 0.89 → 0.92; `NumXMbaGeoShare` 0.54 → 0.61; `NumXMbaGapLo` 0.29 → 0.22; `NumXMbaGapHi` 0.41 → 0.41 |
| Crossover P4T second seed: same direction | ✓ | ✓ | `NumXSeedOneMotShare` 0.73 → 0.65; `NumXSeedOneGeoShare` 0.45 → 0.55 |
| Method ranking consistent across backbones (Kendall τ > 0.5) | ✓ | ✓ | `NumTauPSTPFT` 0.81 → 0.71; `NumTauPSTMBA` 0.71 → 0.81; `NumTauPFTMBA` 0.90 → 0.71 |
| Sanity: after full randomization the KCA rank correlation ≈ 0 (absolute value < 0.2), Grad×Input still > 0.5 | ✓ | ✓ | `NumSanKca` $-$0.01 → $-$0.10; `NumSanGxI` 0.72 → 0.81 |
| Ablation: at m=24 the context estimator repairs the collapse of local necessity | ✓ | ✓ | `NumAblCtxFineDel` 0.228 → 0.212; `NumAblLocalFineDel` 0.432 → 0.334 |
| displace probe: freeze changes the prediction more often than trajectory reversal | ✓ | ✓ | `NumDspFreezeFlip` 48 → 42; `NumDspReverseFlip` 18 → 22 |

Claims that no longer hold at full scale: **4** (✗). Each of them needs the text rewritten, or the experiment re-checked.

## 2. Key numbers

| Quantity | Laptop | Full scale |
|---|---|---|
| P4T accuracy % (`BBaccPFT`) | 90.6 | 89.2 |
| PSTNet accuracy % (`BBaccPST`) | 87.5 | 88.9 |
| Mamba4D accuracy % (`BBaccMBA`) | 83.6 | 88.2 |
| P4T KCA-ctx Del (`NumPFTKcaCtxDel`) | 0.178 | 0.207 |
| P4T region occlusion Del (`NumPFTOcclDel`) | 0.219 | 0.250 |
| P4T IG Del (`NumPFTIGDel`) | 0.268 | 0.283 |
| P4T p(KCA-ctx vs occlusion) (`NumPKcaCtxOcclDel`) | {=}0.0043 | {=}0.1 |
| PSTNet KCA-ctx Del (`NumPSTKcaCtxDel`) | 0.191 | 0.217 |
| PSTNet region occlusion Del (`NumPSTOcclDel`) | 0.222 | 0.252 |
| Mamba4D KCA-ctx Del (`NumMBAKcaCtxDel`) | 0.245 | 0.269 |
| Mamba4D region occlusion Del (`NumMBAOcclDel`) | 0.240 | 0.246 |
| Crossover P4T motion-task share (`NumXMotShare`) | 0.81 | 0.83 |
| Crossover P4T shape-task share (`NumXGeoShare`) | 0.39 | 0.60 |
| Crossover PSTNet motion (`NumXPstMotShare`) | 0.85 | 0.86 |
| Crossover PSTNet shape (`NumXPstGeoShare`) | 0.23 | 0.30 |
| Crossover Mamba4D motion (`NumXMbaMotShare`) | 0.89 | 0.92 |
| Crossover Mamba4D shape (`NumXMbaGeoShare`) | 0.54 | 0.61 |
| P4T KCA-local s/clip (`NumPFTKcaTime`) | 3.0 | 0.5 |
| P4T KCA-ctx s/clip (`NumPFTKcaCtxTime`) | 19.5 | 2.3 |

389 number macros in total, 329 of them changed; present on one side only: 0 laptop, 0 full scale. Full list in `numbers_compare.csv`.

## 3. Text to rewrite (sentences describing the laptop training setup)

These sentences describe the laptop-scale setup (width, number of points, hardware, run time) and must be rewritten by hand for the full-scale numbers:

- `sec/4_experiments.tex:82` — matches: CPU
- `sec/4_experiments.tex:99` — matches: CPU, laptop
- `sec/4_setup.tex:4` — matches: 1024 points
- `sec/4_setup.tex:7` — matches: laptop, reduced width
- `sec/5_conclusion.tex:7` — matches: fall short, laptop, matches its published, reduced width
- `sec/X_suppl.tex:42` — matches: 1024 points, CPU, laptop
- `sec/X_suppl_proofs.tex:20` — matches: 1024 points
- `sec/X_suppl_proofs.tex:26` — matches: 1024 points, 50/30/20, Apple, CPU, Widths are reduced, batch 8, dim 256, laptop

Also: `tab_cost` now holds GPU timings (the laptop version was an M4 CPU), so the Cost paragraph of the main text and the table note in the appendix must change together.



In [23]:
import zipfile
Z = f"{DRIVE_ROOT}/kca_full_outputs.zip"
with zipfile.ZipFile(Z, "w", zipfile.ZIP_DEFLATED) as z:
    for root in (f"{DRIVE_ROOT}/results", CK, PF, REP, LOGS):
        for f in sorted(glob.glob(f"{root}/**/*", recursive=True)):
            if os.path.isfile(f) and not f.endswith(".pth"):
                z.write(f, os.path.relpath(f, DRIVE_ROOT))
print(f"{Z} ({os.path.getsize(Z) / 1e6:.0f} MB): download it from Drive, or uncomment the next line")
# from google.colab import files; files.download(Z)

/content/drive/MyDrive/kca_full/kca_full_outputs.zip (3 MB): download it from Drive, or uncomment the next line


## 9. Bring the results back to the Mac

Download `MyDrive/kca_full/kca_full_outputs.zip`, then:
```bash
mkdir -p ~/kca_work/full && unzip -o ~/Downloads/kca_full_outputs.zip -d ~/kca_work/full
open ~/kca_work/full/report/REPORT.md ~/kca_work/full/paper_full/main.pdf
```
The archive contains: `results/msr/` (full-scale per-clip results), `ckpt/*/summary.json` and the training logs, `paper_full/` (new tables, number macros, figures, PDF) and `report/`.

**Once you decide to adopt the full-scale results**, merge them into the paper:
```bash
cd ~/Documents/4DXAI
cp -R ~/kca_work/full/results/msr code/results/msr_full
cp ~/kca_work/full/paper_full/tables/*.tex paper/tables/
cp ~/kca_work/full/paper_full/fig/*.pdf paper/fig/
bash ~/kca_work/tools/check_paper.sh
```
Then, following `REPORT.md`, rewrite the sentences listed in Section 3 that describe the training setup (width, number of points, hardware, run time), and the claims marked ✗ in Section 1. After that, regenerate the tables and figures from the full-scale results on the Mac:
```bash
cd ~/Documents/4DXAI/code
PYTHONPATH=src ~/kca_work/venv/bin/python -m kca.run.paper_tables --results results/msr_full --paper ../paper --ckpt-root ~/kca_work/full/ckpt --data ~/kca_work/data/msr
PYTHONPATH=src MPLBACKEND=Agg ~/kca_work/venv/bin/python -m kca.run.paper_figures --results results/msr_full --data ~/kca_work/data/msr --fig ../paper/fig
```
`finalize.sh` reads the laptop-scale `results/msr` by default. The full-scale checkpoints (`.pth`) stay on Drive in `ckpt/` and `results/msr/crossover*/`, ready for a rebuttal.